# Vlerësimi: përshkrime në shqip kundrejt anglishtes (PK3)

Ky notebook krahason performancën e modelit përfundimtar (TF-IDF + LinearSVC) kur i njëjti përshkrim simptomash jepet **në anglisht** dhe **në shqip**.

| Sistemi | Rruga |
|---|---|
| **Anglisht – model direkt** | `text_en` → `models/svm_tuned_validation.joblib` (`model.predict`) |
| **Anglisht – API e plotë** | `text_en` → `/predict` i Flask (kontrollet e input-it + SVM + kontrolli i përputhjes) |
| **Shqip – përkthim + API** | `text_sq` → `translateSymptoms` nga `frontend/src/translation/sqToEn.js` (i njëjti kod që përdor aplikacioni) → `/predict` i Flask |

- Përkthimi shqip→anglisht bëhet në **frontend** (jo në Flask). Këtu thirret skedari origjinal `sqToEn.js` përmes Node.js, pa e kopjuar dhe pa e ndryshuar.
- API thirret me `app.test_client()` të Flask: kodi është i njëjti, por nuk duhet të nisësh serverin.
- Modeli, fjalori dhe kodi ekzistues **nuk ndryshohen**.
- Vlerësohen vetëm rreshtat ku `text_sq` është plotësuar me dorë. Anglishtja vlerësohet në të njëjtat rreshta, që krahasimi të jetë i drejtë.

**Kërkesat:** `.venv` i projektit dhe Node.js (≥ 22.15) në PATH.

In [ ]:
import json
import shutil
import subprocess
import sys
import tempfile
from collections import Counter
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

project_path = Path.cwd()
if project_path.name == "notebooks":
    project_path = project_path.parent

CSV_PATH = project_path / "data" / "albanian_test" / "descriptions_sq_en.csv"
RESULTS_DIR = project_path / "results"
TRANSLATOR_JS = project_path / "frontend" / "src" / "translation" / "sqToEn.js"
MODEL_PATH = project_path / "models" / "svm_tuned_validation.joblib"

NO_PRED = "(pa parashikim)"   # rast i refuzuar / pa sëmundje të parashikuar

# Të njëjtat kufij si validateSymptoms() në frontend/src/services/predictionApi.js
MIN_LENGTH, MIN_WORDS, MAX_LENGTH = 15, 3, 5000

pd.set_option("display.max_colwidth", 120)
print("Projekti:", project_path)

## 1. Të dhënat

In [ ]:
# sep=None: e njeh vetë ndarësin ("," ose ";"), nëse CSV ruhet nga Excel.
data = pd.read_csv(CSV_PATH, sep=None, engine="python", encoding="utf-8-sig",
                   keep_default_na=False, dtype=str)
data.columns = [c.strip() for c in data.columns]
expected = ["id", "disease", "text_en", "text_sq"]
missing_cols = [c for c in expected if c not in data.columns]
assert not missing_cols, f"Mungojnë kolonat: {missing_cols}"

data = data.apply(lambda col: col.str.strip())
evaluated = data[data["text_sq"] != ""].reset_index(drop=True)

print(f"Rreshta gjithsej: {len(data)} | me text_sq të plotësuar: {len(evaluated)} "
      f"| sëmundje: {evaluated['disease'].nunique()} nga {data['disease'].nunique()}")
if evaluated.empty:
    raise RuntimeError("Kolona text_sq është bosh. Plotëso përkthimet me dorë në "
                       f"{CSV_PATH} dhe ekzekutoje notebook-un përsëri.")
evaluated.groupby("disease").size().rename("raste").to_frame().T

## 2. Modeli dhe API (Flask, pa server)

In [ ]:
model = joblib.load(MODEL_PATH)
print("Modeli:", model.named_steps["classifier"], "| klasa:", len(model.classes_))

# Importon ml-api/app.py ashtu siç është dhe e thërret /predict pa nisur serverin.
sys.path.insert(0, str(project_path / "ml-api"))
import app as ml_api  # noqa: E402

client = ml_api.app.test_client()

def call_api(text):
    """Kthen (parashikimi, statusi, arsyet) nga /predict."""
    response = client.post("/predict", json={"symptoms": text})
    body = response.get_json(silent=True) or {}
    if response.status_code != 200:
        return NO_PRED, f"api_http_{response.status_code}", body.get("error", "")
    if body.get("status") == "prediction":
        return body["predictedDisease"], "prediction", ""
    return NO_PRED, f"api_{body.get('status', 'unknown')}", ";".join(body.get("reasons", []))

## 3. Përkthyesi shqip→anglisht (kodi origjinal i frontend-it)

In [ ]:
# Node nuk i ngarkon dot vetë import-et pa prapashtesë ("./sqMedicalDictionary")
# dhe JSON-in, që Vite i lejon. Ky skript i vogël (në një dosje të përkohshme)
# vetëm ia mundëson Node-it t'i lexojë; sqToEn.js mbetet i pandryshuar.
BRIDGE_JS = r"""
import { registerHooks } from "node:module";
import { readFileSync, existsSync } from "node:fs";
import { fileURLToPath, pathToFileURL } from "node:url";
registerHooks({
  resolve(specifier, context, next) {
    if (specifier.startsWith(".") && context.parentURL && !/\.[cm]?[jt]sx?$|\.json$/.test(specifier)) {
      const candidate = new URL(specifier + ".js", context.parentURL);
      if (existsSync(fileURLToPath(candidate))) return next(candidate.href, context);
    }
    return next(specifier, context);
  },
  load(url, context, next) {
    if (url.endsWith(".json")) {
      const json = readFileSync(fileURLToPath(url), "utf8");
      return { format: "module", source: `export default ${json};`, shortCircuit: true };
    }
    return next(url, context);
  },
});
const { translateSymptoms } = await import(pathToFileURL(process.argv[2]).href);
const texts = JSON.parse(readFileSync(0, "utf8"));
process.stdout.write(JSON.stringify(texts.map((t) => translateSymptoms(t))));
"""

node = shutil.which("node")
assert node, "Node.js nuk u gjet në PATH. Instaloje (≥ 22.15) dhe rinis Jupyter."

def translate_sq(texts):
    with tempfile.TemporaryDirectory() as tmp:
        bridge = Path(tmp) / "bridge.mjs"
        bridge.write_text(BRIDGE_JS, encoding="utf-8")
        run = subprocess.run([node, str(bridge), str(TRANSLATOR_JS)], input=json.dumps(texts),
                             capture_output=True, text=True, encoding="utf-8", check=False)
    if run.returncode != 0:
        raise RuntimeError(f"Përkthyesi dështoi:\n{run.stderr}")
    return json.loads(run.stdout)

print(translate_sq(["Kam temperaturë dhe kollë prej 3 ditësh."])[0]["english"])

## 4. Ekzekutimi i tre sistemeve

Për shqipen ndiqet e njëjta logjikë si `SymptomAnalysis.jsx`:

1. statusi `unclear` (fjalë të panjohura, mohim i paqartë, pasiguri, asnjë simptomë) → **refuzohet** në përkthim, nuk dërgohet;
2. statusi `confirm` (u korrigjuan gabime shkrimi) → në aplikacion përdoruesi duhet të konfirmojë; këtu **pranohet automatikisht** dhe shënohet te `sq_auto_confirmed`;
3. teksti anglisht kalon kontrollin e gjatësisë së frontend-it, pastaj `/predict`.

In [ ]:
def frontend_length_ok(text):
    cleaned = text.strip()
    return len(cleaned) >= MIN_LENGTH and len(cleaned.split()) >= MIN_WORDS and len(cleaned) <= MAX_LENGTH

rows = []
translations = translate_sq(evaluated["text_sq"].tolist())
en_direct = model.predict(evaluated["text_en"].tolist())

for (_, r), tr, direct in zip(evaluated.iterrows(), translations, en_direct):
    en_pred, en_status, en_reason = call_api(r["text_en"])

    if tr["status"] == "unclear":
        sq_pred, sq_status, sq_reason = NO_PRED, "translation_unclear", ";".join(tr["reasons"])
    elif not frontend_length_ok(tr["english"]):
        sq_pred, sq_status, sq_reason = NO_PRED, "frontend_too_short", ""
    else:
        sq_pred, sq_status, sq_reason = call_api(tr["english"])

    rows.append({
        "id": r["id"], "disease": r["disease"], "text_en": r["text_en"], "text_sq": r["text_sq"],
        "en_direct_pred": str(direct),
        "en_api_pred": en_pred, "en_api_status": en_status, "en_api_reason": en_reason,
        "sq_translation_status": tr["status"],
        "sq_translated_en": tr["english"],
        "sq_unknown_words": ", ".join(tr["unknown"]),
        "sq_corrections": ", ".join(f"{c['from']}→{c['to']}" for c in tr["corrections"]),
        "sq_negated_excluded": ", ".join(e["sq"] for e in tr["excluded"]),
        "sq_auto_confirmed": tr["status"] == "confirm",
        "sq_pred": sq_pred, "sq_status": sq_status, "sq_reason": sq_reason,
    })

results = pd.DataFrame(rows)
for prefix in ["en_direct", "en_api", "sq"]:
    results[f"{prefix}_correct"] = results[f"{prefix}_pred"] == results["disease"]
results[["id", "disease", "en_direct_pred", "en_api_pred", "sq_translation_status", "sq_pred"]].head(10)

## 5. Metrikat dhe tabela krahasuese

- **Accuracy** llogaritet mbi **të gjitha** rastet; një rast i refuzuar numërohet si i gabuar.
- **Precision / Recall / F1 (macro)** llogariten mbi klasat e sëmundjeve. Një rast i refuzuar ul recall-in e klasës së vërtetë, por nuk numërohet si parashikim i gabuar i ndonjë klase tjetër.
- **Accuracy kur ka parashikim**: saktësia vetëm në rastet ku sistemi dha një sëmundje.
- Modeli direkt jep gjithmonë një nga 24 klasat, prandaj nuk ka refuzime.

In [ ]:
SYSTEMS = {
    "en_direct": "Anglisht – model direkt",
    "en_api": "Anglisht – API e plotë",
    "sq": "Shqip – përkthim + API",
}

def metrics(y_true, y_pred):
    labels = sorted((set(y_true) | set(y_pred)) - {NO_PRED})
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels,
                                                  average="macro", zero_division=0)
    answered = [(t, q) for t, q in zip(y_true, y_pred) if q != NO_PRED]
    return {
        "Raste": len(y_true),
        "Accuracy (%)": round(accuracy_score(y_true, y_pred) * 100, 2),
        "Precision macro (%)": round(p * 100, 2),
        "Recall macro (%)": round(r * 100, 2),
        "F1 macro (%)": round(f1 * 100, 2),
        "Pa parashikim / refuzuar": len(y_true) - len(answered),
        "Accuracy kur ka parashikim (%)": (round(sum(t == q for t, q in answered) / len(answered) * 100, 2)
                                           if answered else None),
    }

summary = pd.DataFrame([
    {"Sistemi": name, **metrics(results["disease"].tolist(), results[f"{key}_pred"].tolist())}
    for key, name in SYSTEMS.items()
])
summary

In [ ]:
diff = summary.set_index("Sistemi")
for ref in ["Anglisht – model direkt", "Anglisht – API e plotë"]:
    gap = diff.loc["Shqip – përkthim + API", "Accuracy (%)"] - diff.loc[ref, "Accuracy (%)"]
    print(f"Shqip − {ref}: {gap:+.2f} pp (Accuracy)")

print("\nPse u refuzuan rastet:")
refusals = pd.concat([
    results.loc[results["en_api_pred"] == NO_PRED, ["en_api_status", "en_api_reason"]]
           .set_axis(["Faza", "Arsyeja"], axis=1).assign(Gjuha="Anglisht – API"),
    results.loc[results["sq_pred"] == NO_PRED, ["sq_status", "sq_reason"]]
           .set_axis(["Faza", "Arsyeja"], axis=1).assign(Gjuha="Shqip"),
])
(refusals.groupby(["Gjuha", "Faza", "Arsyeja"]).size().rename("raste").to_frame()
 if not refusals.empty else "Asnjë rast i refuzuar.")

## 6. Raste që dështojnë në shqip por jo në anglisht

Rastet ku modeli direkt në anglisht jep sëmundjen e saktë, por rruga shqip jep sëmundje tjetër ose nuk jep fare. `sq_unknown_words` janë fjalët që përkthyesi nuk i gjeti në fjalorin shqip.

In [ ]:
fail_cols = ["id", "disease", "text_sq", "sq_translation_status", "sq_translated_en",
             "sq_unknown_words", "sq_corrections", "sq_negated_excluded", "sq_status", "sq_reason",
             "sq_pred", "en_api_correct"]
sq_only_failures = results[results["en_direct_correct"] & ~results["sq_correct"]][fail_cols]
print(f"{len(sq_only_failures)} raste dështojnë vetëm në shqip "
      f"(nga {int(results['en_direct_correct'].sum())} që janë të sakta në anglisht).")
sq_only_failures

In [ ]:
# Fjalët e panjohura në të gjitha rastet shqip (sa herë u shfaqën)
unknown_counter = Counter(
    w.strip().lower() for cell in results["sq_unknown_words"] if cell for w in cell.split(",")
)
(pd.DataFrame(unknown_counter.most_common(), columns=["Fjala", "Herë"])
 if unknown_counter else "Asnjë fjalë e panjohur.")

## 7. Ruajtja e rezultateve

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
detail_file = RESULTS_DIR / "albanian_vs_english.csv"
summary_file = RESULTS_DIR / "albanian_vs_english_summary.csv"
results.to_csv(detail_file, index=False, encoding="utf-8-sig")
summary.to_csv(summary_file, index=False, encoding="utf-8-sig")
print("U ruajt:", detail_file, f"({len(results)} raste)")
print("U ruajt:", summary_file)

## 8. Kufizimet

- Tekstet anglisht janë sintetike (`unseen_v1.csv`, të paverifikuara klinikisht); përkthimet shqip janë bërë me dorë nga autorja. Numri i rasteve është i vogël (≤ 72), ndaj diferencat e vogla nuk janë domethënëse statistikisht.
- Rastet `confirm` pranohen automatikisht; në aplikacion përdoruesi duhet t'i konfirmojë.
- Ky vlerësim mat të gjithë rrugën shqip (fjalori + përkthimi + API), jo vetëm modelin. Një rast i refuzuar nuk është gabim i modelit, por mbrojtje e qëllimshme kur teksti nuk kuptohet.